# Day 57 Project — Solution: Deployment-Ready API

**Deliverables:**
- `deploy_api.py` — production API with `/health`, env-var config
- `Procfile` — start command for Render/Railway
- `requirements.txt` — dependency list
- `render.yaml` — Render service manifest

In [ ]:
# ── provided source string ──────────────────────────────────────────────────
_DEPLOY_API_SRC = '"""deploy_api.py — Day 057: deployment-ready Doc-Upload AI API.\n\nConfig from environment variables (set these before running):\n  PORT               - port to bind (default 8000)\n  MODEL              - Ollama model name (default llama3.2)\n  SECRET_KEY         - secret for future auth; set a real value in production\n  MAX_UPLOAD_BYTES   - max upload size in bytes (default 5242880 = 5 MB)\n\nRun:  uvicorn deploy_api:app --host 0.0.0.0 --port ${PORT:-8000}\n"""\nimport io\nimport os\nimport re\nimport secrets\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport pypdf\nfrom fastapi import FastAPI, File, HTTPException, UploadFile\nfrom fastapi.middleware.cors import CORSMiddleware\nimport ollama\n\n# --- config from environment -----------------------------------------------\nPORT            = int(os.environ.get("PORT", "8000"))\nMODEL           = os.environ.get("MODEL", "llama3.2")\nSECRET_KEY      = os.environ.get("SECRET_KEY", "change-me-in-production")\nMAX_SIZE        = int(os.environ.get("MAX_UPLOAD_BYTES", str(5 * 1024 * 1024)))\nMAX_DOC_CHARS   = 4000\nALLOWED_TYPES   = {"text/plain", "application/pdf"}\nUPLOAD_DIR      = Path("uploads")\nAPP_VERSION     = "1.0.0"\n\n# --- helpers ----------------------------------------------------------------\n\ndef validate_upload(content: bytes, filename: str,\n                    allowed_types: set[str], content_type: str,\n                    max_bytes: int) -> tuple[bool, str]:\n    if len(content) == 0:\n        return False, "File is empty"\n    if len(content) > max_bytes:\n        return False, f"File too large ({len(content)} bytes, max {max_bytes})"\n    ext = Path(filename).suffix.lower()\n    if content_type not in allowed_types and ext not in {".txt", ".pdf"}:\n        return False, f"Unsupported type: {content_type}"\n    return True, ""\n\n\ndef safe_filename(original: str) -> str:\n    name = Path(original).name\n    name = re.sub(r"[^\\w\\-.]", "_", name)\n    return f"{secrets.token_hex(4)}_{name}"\n\n\ndef save_upload(content: bytes, filename: str, upload_dir: Path) -> Path:\n    upload_dir.mkdir(parents=True, exist_ok=True)\n    dest = upload_dir / safe_filename(filename)\n    dest.write_bytes(content)\n    return dest\n\n\ndef extract_text(content: bytes, content_type: str) -> str:\n    if "pdf" in content_type:\n        reader = pypdf.PdfReader(io.BytesIO(content))\n        return "\\n".join(p.extract_text() or "" for p in reader.pages)\n    return content.decode("utf-8", errors="replace")\n\n\ndef build_doc_prompt(document_text: str, question: str,\n                     max_doc_chars: int = MAX_DOC_CHARS) -> str:\n    snippet = document_text[:max_doc_chars]\n    return (\n        "You are a helpful assistant. Answer based only on the document below.\\n\\n"\n        f"DOCUMENT:\\n{snippet}\\n\\nQUESTION: {question}"\n    )\n\n# --- in-memory store --------------------------------------------------------\n_docs: dict[str, dict] = {}\n\n# --- app --------------------------------------------------------------------\napp = FastAPI(title="Doc Upload AI API", version=APP_VERSION)\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=False,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\n\n@app.get("/health")\ndef health():\n    return {\n        "status": "ok",\n        "timestamp": datetime.utcnow().isoformat() + "Z",\n        "version": APP_VERSION,\n    }\n\n\n@app.post("/upload", status_code=201)\nasync def upload_doc(file: UploadFile = File(...)):\n    content = await file.read()\n    ok, err = validate_upload(\n        content, file.filename or "unnamed",\n        ALLOWED_TYPES, file.content_type or "", MAX_SIZE\n    )\n    if not ok:\n        raise HTTPException(status_code=400, detail=err)\n    text   = extract_text(content, file.content_type or "text/plain")\n    doc_id = secrets.token_urlsafe(8)\n    _docs[doc_id] = {"filename": file.filename, "text": text}\n    save_upload(content, file.filename or "unnamed", UPLOAD_DIR)\n    return {"doc_id": doc_id, "filename": file.filename, "chars": len(text)}\n\n\n@app.get("/documents")\ndef list_documents():\n    return {"documents": [{"doc_id": k, "filename": v["filename"]}\n                          for k, v in _docs.items()]}\n\n\n@app.post("/ask/{doc_id}")\ndef ask(doc_id: str, question: str):\n    entry = _docs.get(doc_id)\n    if entry is None:\n        raise HTTPException(status_code=404, detail="Document not found")\n    prompt = build_doc_prompt(entry["text"], question)\n    reply  = ollama.chat(\n        model=MODEL,\n        messages=[{"role": "user", "content": prompt}]\n    )["message"]["content"]\n    return {"reply": reply, "doc_id": doc_id}\n\n\nif __name__ == "__main__":\n    import uvicorn\n    uvicorn.run(app, host="0.0.0.0", port=PORT)\n'

from pathlib import Path

def write_deploy_api(path: str) -> str:
    Path(path).write_text(_DEPLOY_API_SRC, encoding="utf-8")
    return path

out = write_deploy_api("deploy_api.py")
print(f"Generated: {out}  ({len(_DEPLOY_API_SRC)} chars)")
print(Path(out).read_text(encoding="utf-8")[:120] + "...")


In [ ]:
# ── generate deployment files ───────────────────────────────────────────────
_PROCFILE_SRC  = 'web: uvicorn deploy_api:app --host 0.0.0.0 --port $PORT\n'
_RENDER_SRC    = 'services:\n  - type: web\n    name: doc-ai-api\n    env: python\n    buildCommand: pip install -r requirements.txt\n    startCommand: uvicorn deploy_api:app --host 0.0.0.0 --port $PORT\n    envVars:\n      - key: MODEL\n        value: llama3.2\n      - key: MAX_UPLOAD_BYTES\n        value: "5242880"\n      - key: SECRET_KEY\n        generateValue: true\n'
_REQS_SRC      = 'fastapi>=0.100.0\nuvicorn[standard]>=0.20.0\npython-multipart>=0.0.5\npypdf>=3.0.0\npython-jose[cryptography]>=3.3.0\nbcrypt>=4.0.0\nollama>=0.1.0\nhttpx>=0.24.0\n'

def write_deployment_files(directory: str) -> dict:
    d = Path(directory)
    d.mkdir(parents=True, exist_ok=True)
    pf = d / "Procfile"
    ry = d / "render.yaml"
    rq = d / "requirements.txt"
    pf.write_text(_PROCFILE_SRC, encoding="utf-8")
    ry.write_text(_RENDER_SRC,   encoding="utf-8")
    rq.write_text(_REQS_SRC,     encoding="utf-8")
    return {"procfile": str(pf), "render_yaml": str(ry), "requirements": str(rq)}

files = write_deployment_files(".")
for name, path in files.items():
    size = Path(path).stat().st_size
    print(f"  wrote {path}  ({size} bytes)")
print()
print("Procfile:")
print(Path(files["procfile"]).read_text())


In [ ]:
# ── smoke-test deploy_api in-process (no Ollama) ────────────────────────────
import io
import re
import secrets
from datetime import datetime
from pathlib import Path
import pypdf
from fastapi import FastAPI, File, HTTPException, UploadFile
from starlette.testclient import TestClient

# --- replicate deploy_api internals in-process ----------------------------

def _validate(content, filename, allowed, ctype, maxb):
    if not content: return False, "empty"
    if len(content) > maxb: return False, "too large"
    ext = Path(filename).suffix.lower()
    if ctype not in allowed and ext not in {".txt", ".pdf"}:
        return False, f"bad type: {ctype}"
    return True, ""

def _extract(content, ctype):
    if "pdf" in ctype:
        r = pypdf.PdfReader(io.BytesIO(content))
        return "\n".join(p.extract_text() or "" for p in r.pages)
    return content.decode("utf-8", errors="replace")

ALLOWED = {"text/plain", "application/pdf"}
MAX_SZ  = 5 * 1024 * 1024
APP_VER = "1.0.0"
_docs: dict = {}

app = FastAPI(title="deploy_api (test)", version=APP_VER)

@app.get("/health")
def health():
    return {"status": "ok",
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "version": APP_VER}

@app.post("/upload", status_code=201)
async def upload(file: UploadFile = File(...)):
    content = await file.read()
    ok, err = _validate(content, file.filename or "unnamed",
                        ALLOWED, file.content_type or "", MAX_SZ)
    if not ok:
        raise HTTPException(400, detail=err)
    text   = _extract(content, file.content_type or "text/plain")
    doc_id = secrets.token_urlsafe(8)
    _docs[doc_id] = {"filename": file.filename, "text": text}
    return {"doc_id": doc_id, "filename": file.filename, "chars": len(text)}

@app.get("/documents")
def list_documents():
    return {"documents": [{"doc_id": k, "filename": v["filename"]}
                          for k, v in _docs.items()]}

# ── run checks ──────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=False)
score = 0; total = 5

def chk(n, ok, msg):
    global score
    print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
    if ok: score += 1

# 1. health → 200 + status=ok
r = client.get("/health")
chk(1, r.status_code == 200 and r.json().get("status") == "ok",
    f"GET /health → 200 + status=ok (got {r.status_code})")

# 2. health has timestamp + version
data = r.json() if r.status_code == 200 else {}
chk(2, "timestamp" in data and "version" in data,
    f"health has timestamp + version (got {list(data.keys())})")

# 3. upload → 201
r2 = client.post("/upload",
                 files={"file": ("readme.txt", b"This is a test document.", "text/plain")})
chk(3, r2.status_code == 201,
    f"POST /upload → 201 (got {r2.status_code})")

up_data = r2.json() if r2.status_code == 201 else {}
doc_id  = up_data.get("doc_id", "")

# 4. GET /documents lists the uploaded file
r3 = client.get("/documents")
docs = r3.json().get("documents", []) if r3.status_code == 200 else []
chk(4, any(d["doc_id"] == doc_id for d in docs),
    f"GET /documents includes uploaded doc (found {len(docs)} docs)")

# 5. invalid upload → 400
r4 = client.post("/upload",
                 files={"file": ("photo.jpg", b"\xff\xd8\xff", "image/jpeg")})
chk(5, r4.status_code == 400,
    f"unsupported type → 400 (got {r4.status_code})")

print(f"\nScore: {score} / {total}")
if score == total:
    print("\nDay 57 — Deploying Apps complete! 🎉")
print(f"\nDeliverable files:")
print("  deploy_api.py   — run with: uvicorn deploy_api:app --reload")
print("  Procfile        — push to Render/Railway")
print("  requirements.txt — dependencies")
print("  render.yaml     — Render service config")
print(f"\nAPI generated: {len(_DEPLOY_API_SRC)} chars")
